# Tutorial: Two-Building District Cooling Network

This notebook walks you through a complete **design + time-series simulation** of a small
district cooling network using the `DisCoolPy` tool.

**What you will learn:**
1. How to define a network configuration (via Python dict or YAML file)
2. How to run a steady-state design solve
3. How to run time-varying offdesign snapshots
4. How to extract and plot key performance indicators

**Network topology:**
```
Chiller → Pump → supply_1 → [Splitter 1] → supply_2 → [Splitter 2] → building_2
                                  ↓                         ↓
                             building_1                  bypass
                                  ↓                         ↓
                            [Merger 1] ← return_2 ← [Merger 2]
                                  ↓
                             return_1 → Chiller
```

> **Note on pressure constraints:** Only buildings that are *not* the last in the chain
> should have `pr` set. The last building is hydraulically short-circuited by the bypass
> valve; setting its `pr` creates a circular dependency. See `docs/pipe_parameter_guide.md`
> for the full explanation.


## 1. Environment Check

Make sure the package is installed. If you cloned the repository, run:
```bash
pip install -e .
```

In [ ]:
import importlib, sys
pkg = 'discoolpy'
if importlib.util.find_spec(pkg) is None:
    raise ImportError(f'{pkg} not found. Run: pip install -e /path/to/repo')
print(f'Package found: {pkg}')
print(f'Python {sys.version}')

## 2. Imports


In [ ]:
import warnings
warnings.filterwarnings('ignore')  # suppress TESPy convergence verbosity

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from discoolpy.utils import (
    build_standard_branch_system,
    load_yaml_config,
)

print('Imports OK')

## 3. Network Configuration

You can define the configuration either as a **Python dictionary** (shown below) or
load it from a **YAML file** (see the commented-out alternative).

The YAML file for this tutorial is at `configs/confige_tutorial.yaml`.


In [ ]:
# ── Option A: Python dict (used in this notebook) ──────────────────────
config = {
    'network': {'chilled_water_cycle_closer': 'chw cycle closer'},
    'design': {
        'supply_temperature_degC': 7.0,
        'building_return_temperature_degC': 12.0,
        'ambient_temperature_degC': 35.0,
        'condenser_inlet_temperature_degC': 30.0,
        'condenser_outlet_temperature_degC': 35.0,
        'chilled_water_pressure_bar': 3.0,
        'condenser_pressure_bar': 3.0,
        'pump_power_W': 300.0,
        'pump_efficiency': 0.70,
        'bypass_mass_flow_kg_s': 0.05,
        'cp_water_J_kgK': 4180.0,
    },
    'chiller': {
        'label': 'tutorial_chiller',
        'T_evap_degC': 2.0,
        'T_cond_degC': 40.0,
        'eta_s': 0.80,
        'refrigerant': 'R134a',
        'native_offdesign': {'enabled': True, 'condenser_ttd_u': 5.0},
    },
    'cooling_tower': {
        'label': 'tutorial_cooling_tower',
        'fluid': {'water': 1.0},
        'approach_temperature_K': -5.0,
        'native_offdesign': False,
        'start_mass_flow_kg_s': 20.0,
    },
    'buildings': [
        # building_1: pr is set (not the last building)
        {'label': 'building_1', 'Q_design_W': 150_000.0, 'pr': 0.995},
        # building_2: pr=None (last building — bypass short-circuit rule)
        {'label': 'building_2', 'Q_design_W': 150_000.0, 'pr': None},
    ],
    'branch': {
        'label': 'tutorial_branch',
        'pump_placement': 'supply_inlet',
        'pump_label': 'main pump',
        'native_offdesign': False,
        'pipe_model': 'pressure_ratio',
        'pipes': {
            'supply_1': {'Q': 0.0},
            'supply_2': {'pr': 0.997, 'Q': 0.0},  # links pressure islands
            'return_2': {'Q': 0.0},
            'return_1': {'pr': 0.999},             # anchors return-header pressure
        },
    },
    'solver': {'iterinfo': False, 'max_iter': 200},
    'outputs': {'output_dir': 'tutorial_outputs'},
}

# ── Option B: load from YAML ─────────────────────────────────────────────
# config = load_yaml_config('../configs/config_tutorial.yaml')

print('Configuration ready.')
print(f"  Buildings: {[b['label'] for b in config['buildings']]}")
print(f"  Refrigerant: {config['chiller']['refrigerant']}")
print(f"  Supply temperature: {config['design']['supply_temperature_degC']} °C")

## 4. Build the TESPy Network

`build_standard_branch_system` assembles the full TESPy network — chiller, cooling tower,
distribution branch, buildings, and hydraulic connections — and returns the component objects
you need for post-processing.


In [ ]:
nw, chiller, buildings, branch, cooling_tower, c_chiller_to_cc = \
    build_standard_branch_system(config)

print('Network built.')
print(f'  Components: {len(list(nw.comps.index))} TESPy components')
print(f'  Connections: {len(list(nw.conns.index))} TESPy connections')

## 5. Design-Point Solve

The design solve establishes all component parameters at the rated operating point.
TESPy saves the design state to disk so that offdesign snapshots can reference it.


In [ ]:
DESIGN_PATH = 'tutorial_design_state'

nw.solve(mode='design', max_iter=config['solver']['max_iter'])
assert nw.converged, 'Design solve did not converge!'
nw.save(DESIGN_PATH)

# ── Print design-point summary ────────────────────────────────────────
Q_evap = chiller.evaporator.Q.val          # W  (negative = heat removal)
W_comp = chiller.compressor.P.val          # W
COP    = abs(Q_evap) / W_comp

print('Design-point results')
print(f'  Total cooling load : {abs(Q_evap)/1e3:.1f} kW')
print(f'  Compressor power   : {W_comp/1e3:.2f} kW')
print(f'  COP                : {COP:.2f}')
print(f'  Pump power         : {branch.pump.P.val:.1f} W')
print()
for b in buildings:
    print(f'  {b.label}: Q = {abs(b.heat_exchanger.Q.val)/1e3:.1f} kW, '
          f'pr = {b.heat_exchanger.pr.val:.4f}')

## 6. Build a Dummy Hourly Load Profile

We create a synthetic 24-hour cooling load profile for each building using a
sinusoidal pattern that peaks at 14:00 and reaches a minimum at 04:00 — a
reasonable approximation of commercial building cooling demand in a hot climate.


In [ ]:
hours = np.arange(24)

# Fraction of design load at each hour (0.3 – 1.0)
load_fraction = 0.35 + 0.65 * np.clip(
    np.sin(np.pi * (hours - 4) / 20), 0, 1
)

# Building 2 has a slightly different profile (e.g. residential vs commercial)
load_fraction_b2 = 0.30 + 0.70 * np.clip(
    np.sin(np.pi * (hours - 6) / 18), 0, 1
)

Q_design_b1 = config['buildings'][0]['Q_design_W']
Q_design_b2 = config['buildings'][1]['Q_design_W']

profile = pd.DataFrame({
    'hour': hours,
    'Q_b1_W': Q_design_b1 * load_fraction,     # positive = cooling demand (W)
    'Q_b2_W': Q_design_b2 * load_fraction_b2,  # positive = cooling demand (W)
})

# Plot the profile
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(hours, profile['Q_b1_W'] / 1e3, label='Building 1', marker='o', ms=4)
ax.plot(hours, profile['Q_b2_W'] / 1e3, label='Building 2', marker='s', ms=4)
ax.set_xlabel('Hour of day')
ax.set_ylabel('Cooling load (kW)')
ax.set_title('Dummy 24-hour cooling load profile')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Time-Varying Offdesign Simulation

We iterate over each hour, update the building loads and mass flows, then call
`nw.solve(mode='offdesign', ...)` to compute the system state at that operating point.


In [ ]:
cp   = config['design']['cp_water_J_kgK']
dT   = (config['design']['building_return_temperature_degC']
        - config['design']['supply_temperature_degC'])  # 5 K

results = []

for _, row in profile.iterrows():
    h = int(row['hour'])
    Q_vals = [abs(row['Q_b1_W']), abs(row['Q_b2_W'])]  # positive W

    # Update each building's heat load and mass flow
    for b, Q in zip(buildings, Q_vals):
        m = abs(Q) / (cp * dT)
        b.set_demand(Q)
        b.inlet.set_attr(m=m)

    nw.solve(mode='offdesign', design_path=DESIGN_PATH,
             max_iter=config['solver']['max_iter'])

    if nw.converged:
        Q_evap = chiller.evaporator.Q.val
        W_comp = chiller.compressor.P.val
        COP    = abs(Q_evap) / W_comp if W_comp > 0 else float('nan')
        results.append({
            'hour'   : h,
            'Q_evap_kW': abs(Q_evap) / 1e3,
            'W_comp_kW': W_comp / 1e3,
            'COP'    : COP,
            'converged': True,
        })
    else:
        results.append({'hour': h, 'Q_evap_kW': None,
                        'W_comp_kW': None, 'COP': None, 'converged': False})
        print(f'  Hour {h:02d}: did not converge')

df = pd.DataFrame(results)
print(f'Converged: {df["converged"].sum()} / {len(df)} hours')
df[['hour', 'Q_evap_kW', 'W_comp_kW', 'COP']].round(2)

## 8. Results Visualisation


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True)

axes[0].plot(df['hour'], df['Q_evap_kW'], color='steelblue', marker='o', ms=4)
axes[0].set_ylabel('Chiller load (kW)')
axes[0].set_title('Time-varying district cooling simulation — 2-building tutorial')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df['hour'], df['W_comp_kW'], color='tomato', marker='s', ms=4)
axes[1].set_ylabel('Compressor power (kW)')
axes[1].grid(True, alpha=0.3)

axes[2].plot(df['hour'], df['COP'], color='seagreen', marker='^', ms=4)
axes[2].set_ylabel('COP  (—)')
axes[2].set_xlabel('Hour of day')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
Path('tutorial_outputs').mkdir(exist_ok=True)
plt.savefig('tutorial_outputs/tutorial_results.png', dpi=150)
plt.show()
print('Figure saved to tutorial_outputs/tutorial_results.png')

## 9. Summary Statistics


In [ ]:
summary = df[['Q_evap_kW', 'W_comp_kW', 'COP']].describe().round(3)
print(summary)

print(f"\nPeak cooling load : {df['Q_evap_kW'].max():.1f} kW")
print(f"Mean COP          : {df['COP'].mean():.3f}")
print(f"Min COP           : {df['COP'].min():.3f}")
print(f"Max COP           : {df['COP'].max():.3f}")

## 10. Using a YAML Configuration File

For real projects, store your configuration in a YAML file and load it with
`load_yaml_config`. This separates data from code and makes scenarios easy to
version-control and share.

```python
from discoolpy.utils import load_yaml_config

config = load_yaml_config('../configs/config_tutorial.yaml')
nw, chiller, buildings, branch, cooling_tower, c_chiller_to_cc = \
    build_standard_branch_system(config)
```

The YAML file for this tutorial is at `configs/config_tutorial.yaml`.
The other three-building planned-length scenario is at
`configs/config_length_derived_pr.yaml`.


## 11. Next Steps

| Topic | File |
|---|---|
| Add a cold/ice storage tank | `examples/storage_comparison_example.py` |
| Three-building planned-length scenario | `configs/config_length_derived_pr.yaml` |
| Pipe parameter selection guide | `docs/pipe_parameter_guide.md` |
| Full API reference | `docs/api_reference.md` |
| Getting started guide | `docs/getting_started.md` |
